In [1]:
import pandas as pd

URL = "https://raw.githubusercontent.com/yamanalab/ec-darkpattern/master/dataset/dataset.tsv"
df = pd.read_csv(URL, sep="\t")
print(df.shape)
print(df.columns.tolist())
df.head(10)

(2356, 4)
['page_id', 'text', 'label', 'Pattern Category']


,page_id,text,label,Pattern Category
0,1012,FLASH SALE | LIMITED TIME ONLY Shop Now,1,Urgency
1,158,Pillowcases & Shams,0,Not Dark Pattern
2,108,Write a review,0,Not Dark Pattern
3,1425,"To start your return, simply click on the foll...",0,Not Dark Pattern
4,1658,newsletter signup (privacy policy),0,Not Dark Pattern
5,975,In demand,1,Scarcity
6,1469,We cannot guarantee door to door delivery for ...,0,Not Dark Pattern
7,1570,Hurry! Only 2 left in stock,1,Scarcity
8,1617,In Stock only 3 left,1,Scarcity
9,108,International Shipping Policy,0,Not Dark Pattern


In [2]:
TEXT_COL = "text"    # change if Cell 1 showed a different column name for the text
LABEL_COL = "label"  # change if Cell 1 showed a different column name for dark / not dark

df = df[[TEXT_COL, LABEL_COL]].dropna()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL] != ""]

before = len(df)
df = df.drop_duplicates(subset=TEXT_COL)   # duplicates in train AND test would inflate your scores
print("duplicates removed:", before - len(df))

df[LABEL_COL] = df[LABEL_COL].astype(int)
print(df[LABEL_COL].value_counts())

# Check which label means "dark": read a few examples of each
for lab in sorted(df[LABEL_COL].unique()):
    print("\nLabel", lab, "examples:")
    print(df[df[LABEL_COL] == lab][TEXT_COL].head(5).to_string())

# Compare text lengths (in words) between the two groups
df["words"] = df[TEXT_COL].str.split().str.len()
print("\nWords per text, by label:")
print(df.groupby(LABEL_COL)["words"].describe())

duplicates removed: 7
label
1    1178
0    1171
Name: count, dtype: int64

Label 0 examples:
1                                  Pillowcases & Shams
2                                       Write a review
3    To start your return, simply click on the foll...
4                   newsletter signup (privacy policy)
6    We cannot guarantee door to door delivery for ...

Label 1 examples:
0     FLASH SALE | LIMITED TIME ONLY Shop Now
5                                   In demand
7                 Hurry! Only 2 left in stock
8                        In Stock only 3 left
11                         Hurry! Only 2 left

Words per text, by label:
        count      mean        std  min  25%  50%  75%    max
label                                                        
0      1171.0  5.764304   8.852879  1.0  2.0  3.0  6.0  116.0
1      1178.0  9.105263  12.892568  1.0  4.0  6.0  9.0  143.0


In [3]:
from sklearn.model_selection import train_test_split

X = df[TEXT_COL]
y = df[LABEL_COL]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("train:", len(X_train), " validation:", len(X_val), " test:", len(X_test))

train: 1644  validation: 352  test: 353


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(X_train, y_train)
print("trained")

trained


In [5]:
from sklearn.metrics import classification_report, confusion_matrix

val_pred = model.predict(X_val)
print(classification_report(y_val, val_pred, digits=3))
print("Confusion matrix (rows = truth, columns = prediction):")
print(confusion_matrix(y_val, val_pred))

              precision    recall  f1-score   support

           0      0.876     0.971     0.921       175
           1      0.968     0.864     0.913       177

    accuracy                          0.918       352
   macro avg      0.922     0.918     0.917       352
weighted avg      0.923     0.918     0.917       352

Confusion matrix (rows = truth, columns = prediction):
[[170   5]
 [ 24 153]]


In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score

probs = model.predict_proba(X_val)[:, 1]
print("threshold  precision  recall   f1")
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pred = (probs >= t).astype(int)
    print(f"{t:.1f}        {precision_score(y_val, pred):.3f}      "
          f"{recall_score(y_val, pred):.3f}    {f1_score(y_val, pred):.3f}")

threshold  precision  recall   f1
0.3        0.860      0.938    0.897
0.4        0.920      0.915    0.918
0.5        0.968      0.864    0.913
0.6        0.993      0.825    0.901
0.7        1.000      0.751    0.858
0.8        1.000      0.627    0.771
0.9        1.000      0.379    0.549


In [7]:
THRESHOLD = 0.7   # replace with the value you chose in Cell 6

test_probs = model.predict_proba(X_test)[:, 1]
test_pred = (test_probs >= THRESHOLD).astype(int)

print(classification_report(y_test, test_pred, digits=3))
cm = confusion_matrix(y_test, test_pred)
print(cm)
tn, fp, fn, tp = cm.ravel()
print("false positive rate:", round(fp / (fp + tn), 3))

              precision    recall  f1-score   support

           0      0.825     0.989     0.899       176
           1      0.986     0.791     0.878       177

    accuracy                          0.890       353
   macro avg      0.905     0.890     0.888       353
weighted avg      0.906     0.890     0.888       353

[[174   2]
 [ 37 140]]
false positive rate: 0.011


In [8]:
samples = [
    "Add to cart", "Contact us", "Sign in", "No thanks", "Your cart is empty",
    "Free delivery on orders above 499", "Subscribe to our newsletter",
    "Only 2 left in stock!", "Hurry! Offer ends in 09:59",
    "12 people are viewing this right now",
    "No thanks, I hate saving money", "Yes, send me offers from partners",
]
for s, p in zip(samples, model.predict_proba(samples)[:, 1]):
    print(f"{p:.2f}  {s}")

0.36  Add to cart
0.27  Contact us
0.28  Sign in
0.94  No thanks
0.36  Your cart is empty
0.37  Free delivery on orders above 499
0.19  Subscribe to our newsletter
0.99  Only 2 left in stock!
0.89  Hurry! Offer ends in 09:59
0.86  12 people are viewing this right now
0.78  No thanks, I hate saving money
0.32  Yes, send me offers from partners


In [9]:
import joblib
import sklearn
from google.colab import files

print("scikit-learn version:", sklearn.__version__)   # write this down
joblib.dump({"pipeline": model, "threshold": THRESHOLD}, "model.joblib")
files.download("model.joblib")

scikit-learn version: 1.6.1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>